In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
df.shape

(100, 6)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ["has_covid"]), df["has_covid"], test_size=0.2, random_state=1)

In [5]:
X_train.shape

(80, 5)

Without Column Transformer

In [6]:
# Simple Inputer to fill missing values
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer()

X_train_fever = imputer.fit_transform(X_train[["fever"]])

X_test_fever = imputer.transform(X_test[["fever"]])

X_train_fever.shape

(80, 1)

In [8]:
# ordinal encoding for cough column

df["cough"].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [9]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[["Mild", "Strong"]])

X_train_cough = oe.fit_transform(X_train[["cough"]])

X_test_cough = oe.transform(X_test[["cough"]])

X_train_cough.shape

(80, 1)

In [11]:
# One hot encoding for gender and city

from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop="first", sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])

X_test_gender_city = ohe.transform(X_test[["gender", "city"]])

X_train_gender_city.shape

(80, 4)

In [12]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [13]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

Column Transformer

In [17]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ("tf1", SimpleImputer(), ["fever"]),
    ("tf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
    ("tf3", OneHotEncoder(drop="first", sparse_output=False), ["gender", "city"])
], remainder="passthrough")

In [18]:
X_train_new = transformer.fit_transform(X_train)

X_train_new.shape

(80, 7)

In [19]:
X_test_new = transformer.transform(X_test)

X_test_new.shape

(20, 7)

In [20]:
X_test_new

array([[ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  69.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  26.        ],
       [ 99.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  65.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  27.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  40.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  38.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  24.        ],
       [103.        ,   0.        ,   0.        ,   1.        ,
          0.    